In [21]:
import copy
import numpy as np
import pickle
import copy
from all_functions import *
from chart_utils import make_plots
from utils import create_graphs
import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt
import collections
from PIL import Image
import networkx as nx
%load_ext autoreload
%autoreload 2

def evaluate_agent(agent, env, start_state, goal_state, max_time_steps):
    eval_return = 0
    state = env.reset_world(start_state, goal_state)
    for t in range(max_time_steps):
        action = agent.option_action(state, False)
        ns, r, done = env.step(action)
        eval_return += r
        state = ns
        if done:
            print('done..')
            break
    return eval_return

def get_start_and_goal_state(run,gw,G):
    np.random.seed(run)
    print('finding start and goal state')
    goal_state = np.random.randint(0, len(G.nodes))
    print('goal: ', goal_state)
    start_state = -1
    max_spd = -999999999
    for _node in gw.G.nodes:
                                spd = nx.shortest_path_length(gw.G, source=_node, target=goal_state)
                                if spd > max_spd:
                                     start_state = _node
                                     max_spd = spd    
    print('max_spd:',max_spd, 'start_state: ', start_state)
    return start_state, goal_state

def run_training_on_env(env_name, option_name, num_options=4, instances=5, max_steps_mul=250, ep_time_horizon=1000, eval_interval=2000, directed=False, option_eps=0.1, nsa_greedy=False, det_option_policy=True, reward_eval=True, RW_LAP=True, beta=0.5,k_val=64):
    max_steps_mul = 250
    stochastic = False
    # 1) Create the environment
    grid_world = True
    skip_graph_loader = False
    if env_name == 'double_large_maze_dir':
        directed = True
        env_name = 'double_large_maze'
        skip_graph_loader = True
        G = image_to_graph_loader('double_large_maze')
        grid_world = False
        G = G.to_undirected() 
        G = G.to_directed()
        G.add_edge(351, 5352)
    if env_name == 'DoubleHex_dir':
        directed = True
        env_name = 'DoubleHex'
        skip_graph_loader = True
        G = image_to_graph_loader('DoubleHex')
        grid_world = False
        G = G.to_undirected() 
        G = G.to_directed()
        G.add_edge(696, 4645)
        
    if env_name == 'quad_maze' or env_name == 'office' or env_name == 'Hex' or env_name == 'double_office'  or env_name == 'double_large_maze' or env_name == 'large_maze'  or env_name == 'DoubleHex' or env_name == 'Hex_half' or env_name == 'Office_half':
            if skip_graph_loader == False:
                G = image_to_graph_loader(env_name)
                grid_world = False
                # u, v = find_furthest_states(G)
                # print(f"The two furthest‐apart nodes are {u} and {v} (distance = {nx.shortest_path_length(G, u, v)})")
    elif env_name == 'toh':
        G =  get_tow_graph()
        grid_world = False
    elif env_name == 'torus':
        G = torus()
        grid_world = False
        directed = True
    elif env_name == 'rubiks':
        G = rubiks()
        grid_world = False
    elif env_name == 'four_rooms_dir':
        G = four_rooms_directed()
        directed = True
        grid_world = False
    else:
        G, G2 = create_graphs(env_name)
    if directed == False:
        G = G.to_undirected()
    G = renumber_graph(G)
    pos = nx.get_node_attributes(G,'pos')
    gw = GraphWorld(grid_world, G, stochastic)
    ep_time_horizon = len(G.nodes)
    update_exploration_option_rate = len(G.nodes)*5
    if update_exploration_option_rate < 1000: 
        update_exploration_option_rate = 1000
    if len(G.nodes) < 1000: #smaller graph's should need less time
         ep_time_horizon = 500
         max_steps_mul = max_steps_mul * ep_time_horizon #ep_time_horizon
    elif len(G.nodes) < 3000: 
         ep_time_horizon = 1000
         max_steps_mul = max_steps_mul * ep_time_horizon #ep_time_horizon
    elif len(G.nodes) >= 6000: 
         ep_time_horizon = len(G.nodes)
         max_steps_mul = max_steps_mul * 1000 #ep_time_horizon
    elif len(G.nodes) >= 3000: 
         ep_time_horizon = len(G.nodes)
         max_steps_mul = max_steps_mul * 1000 #ep_time_horizon
            
    print('max_steps_mul: ', max_steps_mul)
    if option_name == 'Qlearning-novelty' or option_name == 'Qlearning':
        option_limit = 0
    else:
        option_limit = 64
    agent = AgentQ2(G, gw, option_limit, policy="epsilon_greedy", epsilon=0.1, alpha=0.4, gamma=0.99,initialization_states=None, option_eps = option_eps, det_option_policy=det_option_policy)
    if reward_eval:
        agent.update_options = True #-------------------------update
    else:
        agent.update_options = False 
    instance_list     = []  # shape: [instances][episodes]
    eval_instance_list = [] # shape: [instances][# of evaluations]
    node_instance_list = []
    edge_instance_list = []
    info_instance_list = []
    for run in range(instances):
        print('len(G.nodes()): ', len(G.nodes()))
        visitation_mat = np.zeros(len(G.nodes()))
        visitation_mat_2 = np.zeros(len(G.nodes()))
        visitation_sa = np.zeros(shape=(gw.num_nodes, gw.max_action_size))
        #go through statespace and set all q_values beyond the number of neighbors to -1
        for node in G.nodes:
            mas = len(gw.get_next_states(node))
            visitation_sa[node][mas: ] = -1
        new_G = nx.DiGraph()
        # Make a fresh copy of 'agent' for each run
        running_agent = copy.deepcopy(agent)
        start_state, goal_state = get_start_and_goal_state(run,gw,G)  #switched
        step_count = 0
        run_rewards = []
        run_eval_rewards = []
        node_counts = []
        edge_counts = []
        info = []
        found_goal = False
        prior_state = start_state
        while(step_count < max_steps_mul):
            state = gw.reset_world(start_state,goal_state) #1500 (large)
            e_rewards = []
            backward_updates_data = []
            for t in range(ep_time_horizon):
                running_agent.visitation_mat = visitation_mat #or step_count == ep_time_horizon
                if ((step_count % update_exploration_option_rate == 0 and step_count > 1000)) and option_name != 'Qlearning' and option_name != 'Qlearning-novelty' and option_name != 'Qlearning-novelaction':
                    replace = False
                    # if num_options == 1:
                    #     replace = True
                    _e, count_vec = running_agent.add_option(directed, new_G,visitation_mat, option_name, num_options, replace, option_limit,RW_LAP,beta=beta,k_val=k_val)
                    if count_vec is not None:
                        q_levels = np.linspace(0, 1, 21)
                        quantile_values = np.quantile(count_vec, q_levels)
                    else:
                        quantile_values = None
                    info.append([_e,quantile_values,step_count])
                    #check_reals(_e,count_vec)
                visitation_mat[state] += 1
                action = running_agent.option_action(state, True) 
                ns, reward, done = gw.step(action)
                visitation_sa[state][action] += 1
                #reward += -1
                if option_name == 'Qlearning-novelty':
                    reward += 0.01/(np.sqrt(visitation_sa[state][action])+0.0000000000000001)
                backward_updates_data.append([state, ns, reward, done, action])
                
                if reward_eval:
                    running_agent.update_q_values(state, ns, reward, done, action) #-------------------------update
                e_rewards.append(reward)
                    #add edge to new graph
                if state not in new_G.nodes():
                    new_G.add_node(state, pos = pos[state])
                if ns not in new_G.nodes():
                    new_G.add_node(ns, pos = pos[ns])
                if not new_G.has_edge(state, ns) and state != ns:# and G.has_edge(state, ns):
                    new_G.add_edge(state, ns)
                state = ns
                if step_count % eval_interval == 0:
                    print('step_count: ', step_count, 'number of nodes: ', len(new_G.nodes()) ,'/', len(G.nodes()), 'number of edges: ', len(new_G.to_undirected().edges()) ,'/', len(G.edges()))
                    eval_ret = evaluate_agent(copy.deepcopy(running_agent), copy.deepcopy(gw), start_state, goal_state, ep_time_horizon)
                    run_eval_rewards.append(eval_ret)
                    node_counts.append(len(new_G.nodes()))
                    edge_counts.append(len(new_G.edges()))

                step_count += 1

                if done:
                    if not found_goal and reward > 0:
                        found_goal = True
                        print("found goal ", step_count)
                        print('_', running_agent.option_eps)
                        if option_name == 'Qlearning-novelty':
                            epsilon = 0.1
                    if reward_eval:
                        break #-------------------------update
                if step_count >= max_steps_mul:
                    break
            if reward_eval:
                # #in reverse
                for i in range(len(backward_updates_data)-1, -1, -1):
                            old_state, new_state, reward, done, action = backward_updates_data[i]
                            running_agent.update_q_values(old_state, new_state, reward, done, action) #-------------------------update
                
            run_rewards.append(np.sum(e_rewards))
        instance_list.append(run_rewards)
        eval_instance_list.append(run_eval_rewards)
        node_instance_list.append(node_counts)
        edge_instance_list.append(edge_counts)
        info_instance_list.append(info)
    #return instance_list, eval_instance_list, running_agent
        print('len: ', len(new_G.nodes()))
    return instance_list, eval_instance_list, node_instance_list, edge_instance_list, running_agent, visitation_mat, info_instance_list

def main():
    #env_names = ['DoubleHex_dir','four_rooms_dir','torus','double_large_maze_dir','four_rooms_11_11.data','toh','office','DoubleHex','double_large_maze','rubiks']
    env_names = ['DoubleHex','double_large_maze']
    option_sets = [4]
    reward_eval_set = [True]
    option_names = ['hotspot_options','cover_options','eigen_options','oracle']#Note: hotspot_options = NEO, oracle = SPNovelty
    nsa_greedy = False
    stochastic = False
    det_option_policy = True
    RW_LAP = True
    results = {'env_results': {}}
    option_eps_set = [0.1]
    beta_set = [0.5]
    k_set = [64]
    for beta in beta_set:
        for k_val in k_set:
            for reward_eval in reward_eval_set:
                for option_name in option_names:
                    for option_eps in option_eps_set:
                        for num_options in option_sets:
                                for env_name in env_names:
                                    print(f"Running training on environment: {env_name}")
                                    print('option_eps: ', option_eps)
                                    # Train & get results for each environment
                                    instance_list, eval_instance_list, node_instance_list, edge_instance_list, last_agent, visitation_mat, info_instance_list = run_training_on_env(env_name, option_name=option_name, num_options=num_options, directed=False, option_eps=option_eps, nsa_greedy=nsa_greedy, det_option_policy=det_option_policy, reward_eval=reward_eval, RW_LAP=RW_LAP, beta=beta, k_val=k_val)
                                    results['env_results'][env_name] = {'node_instance_list:': node_instance_list, 'edge_instance_list': edge_instance_list, 'stochastic': stochastic, 'option_name': option_name,'num_options': num_options, 'training_returns': instance_list,'eval_returns': eval_instance_list,'final_agent': last_agent,'visitation_mat': visitation_mat,'info_instance_list': info_instance_list}
                        
                                    for k in range(len(eval_instance_list)):
                                        print('eval_instance_list: ', np.shape(instance_list[k]))
                                    alg = []
                                    alg.append(node_instance_list)
                                    #UNCOMMENT TO PLOT LEARNING CURVE
                                    plt.rcParams['figure.figsize'] = [5, 5]
                                    make_plots(alg, [option_name], cumulative=False, episodic=False, track_disc_reward=False, open_plot=False)
                                    # # ------------------------------------------
                                    # # SAVE RESULTS TO FILE
                                    # # ------------------------------------------
                                    save_name = 'RESULTS_' + option_name + str(beta) + '_' + str(k_val) + '_' + str(RW_LAP) + '_' + str(nsa_greedy) + '_' + str(det_option_policy) + '__' + str(num_options) + '_' + str(option_eps) + '.pkl'
                                    with open(save_name, "wb") as f:
                                        pickle.dump(results, f)
        
if __name__ == "__main__":
    main()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Running training on environment: DoubleHex
option_eps:  0.1
(95, 78, 4)
max_steps_mul:  250000
len(G.nodes()):  5334
finding start and goal state
goal:  2732
max_spd: 140 start_state:  733
step_count:  0 number of nodes:  2 / 5334 number of edges:  1 / 8346
step_count:  2000 number of nodes:  57 / 5334 number of edges:  86 / 8346
step_count:  4000 number of nodes:  95 / 5334 number of edges:  134 / 8346
step_count:  6000 number of nodes:  150 / 5334 number of edges:  216 / 8346
step_count:  8000 number of nodes:  185 / 5334 number of edges:  258 / 8346
step_count:  10000 number of nodes:  389 / 5334 number of edges:  553 / 8346
step_count:  12000 number of nodes:  427 / 5334 number of edges:  619 / 8346
step_count:  14000 number of nodes:  427 / 5334 number of edges:  620 / 8346
step_count:  16000 number of nodes:  427 / 5334 number of edges:  620 / 8346
step_count:  18000 number of nodes:  472 / 53

RuntimeError: Failed to process string with tex because latex could not be found

<Figure size 500x500 with 1 Axes>

Running training on environment: double_large_maze
option_eps:  0.1
(115, 63, 4)
max_steps_mul:  250000
len(G.nodes()):  5363
finding start and goal state
goal:  2732
max_spd: 140 start_state:  351
step_count:  0 number of nodes:  2 / 5363 number of edges:  1 / 9010
step_count:  2000 number of nodes:  125 / 5363 number of edges:  203 / 9010
step_count:  4000 number of nodes:  161 / 5363 number of edges:  265 / 9010
step_count:  6000 number of nodes:  412 / 5363 number of edges:  622 / 9010
step_count:  8000 number of nodes:  435 / 5363 number of edges:  676 / 9010
step_count:  10000 number of nodes:  441 / 5363 number of edges:  691 / 9010
step_count:  12000 number of nodes:  441 / 5363 number of edges:  691 / 9010
step_count:  14000 number of nodes:  441 / 5363 number of edges:  691 / 9010
step_count:  16000 number of nodes:  441 / 5363 number of edges:  691 / 9010
step_count:  18000 number of nodes:  442 / 5363 number of edges:  696 / 9010
step_count:  20000 number of nodes:  445 / 5

RuntimeError: Failed to process string with tex because latex could not be found

...

Running training on environment: double_large_maze
option_eps:  0.1
(115, 63, 4)
max_steps_mul:  250000
len(G.nodes()):  5363
finding start and goal state
goal:  2732
max_spd: 140 start_state:  351
step_count:  0 number of nodes:  2 / 5363 number of edges:  1 / 9010
step_count:  2000 number of nodes:  125 / 5363 number of edges:  203 / 9010
step_count:  4000 number of nodes:  161 / 5363 number of edges:  265 / 9010
step_count:  6000 number of nodes:  412 / 5363 number of edges:  622 / 9010
step_count:  8000 number of nodes:  435 / 5363 number of edges:  676 / 9010
step_count:  10000 number of nodes:  441 / 5363 number of edges:  691 / 9010
step_count:  12000 number of nodes:  441 / 5363 number of edges:  691 / 9010
step_count:  14000 number of nodes:  441 / 5363 number of edges:  691 / 9010
step_count:  16000 number of nodes:  441 / 5363 number of edges:  691 / 9010
step_count:  18000 number of nodes:  442 / 5363 number of edges:  696 / 9010
step_count:  20000 number of nodes:  445 / 5

<class 'networkx.utils.decorators.argmap'> compilation 43:3: FutureWarning: 

single_target_shortest_path_length will return a dict instead of
an iterator in version 3.5


step_count:  30000 number of nodes:  564 / 5363 number of edges:  892 / 9010
step_count:  32000 number of nodes:  586 / 5363 number of edges:  930 / 9010
step_count:  34000 number of nodes:  620 / 5363 number of edges:  986 / 9010
step_count:  36000 number of nodes:  629 / 5363 number of edges:  1002 / 9010
step_count:  38000 number of nodes:  629 / 5363 number of edges:  1004 / 9010
step_count:  40000 number of nodes:  638 / 5363 number of edges:  1018 / 9010
step_count:  42000 number of nodes:  638 / 5363 number of edges:  1018 / 9010
step_count:  44000 number of nodes:  679 / 5363 number of edges:  1089 / 9010
step_count:  46000 number of nodes:  679 / 5363 number of edges:  1089 / 9010
step_count:  48000 number of nodes:  679 / 5363 number of edges:  1089 / 9010
step_count:  50000 number of nodes:  679 / 5363 number of edges:  1089 / 9010
step_count:  52000 number of nodes:  679 / 5363 number of edges:  1089 / 9010
is_directed: False
step_count:  54000 number of nodes:  689 / 5363 

RuntimeError: Failed to process string with tex because latex could not be found

<Figure size 500x500 with 1 Axes>